# SPARK SESSION

In [ ]:
import findspark
import pandas as pd
findspark.init ()
import pyspark
from pyspark.sql import SparkSession

spark= SparkSession.builder\
              .config('spark.executor.instances','20')\
              .config('spark.driver.memory','5g')\
              .config('spark.executor.memory','20g')\
              .config('spark.executor.cores','5')\
              .config('spark.executor.memoryOverhead','4g')\
              .appName("AML_KA")\
              .master("yarn")\
              .getOrCreate()

# PLDLIB

`pldlib` supplies the standard patient-level-data metrics this notebook builds lines of
therapy from — `stencil.sob`, `stencil.episode`, `stencil.regimen`.

The library is vendored at `vendor/pldlib.egg` and shipped to the executors by
`vendor/spark_pldlib.py`, so the notebook does not need a copy of the egg in the
directory it was launched from. Provenance, version and upgrade steps:
`vendor/README.md`.

In [ ]:
import os
import sys

# Notebooks have no __file__: walk up from the working directory to the repo root.
_repo_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_repo_root, "AGENTS.md")):
    _parent = os.path.dirname(_repo_root)
    if _parent == _repo_root:
        raise RuntimeError(
            "run this notebook from inside the apex-oncology checkout "
            "(no AGENTS.md found above %s)" % os.getcwd()
        )
    _repo_root = _parent

# vendor/ holds the egg and its loader; scripts/ holds dos_grace_config, imported later.
for _path in (
    os.path.join(_repo_root, "vendor"),
    os.path.join(_repo_root, "franchises", "Venclexta", "scripts"),
):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import spark_pldlib

spark_pldlib.attach(spark)          # ships the egg to the executors
print(spark_pldlib.describe())      # confirm the version in use

from pldlib import stencil

# GLOBAL VARIABLES

In [ ]:
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS' 

# HCP-CHILD
reltio_tbl = 'abv_mhcd_synd.mhcd_rltn_all_org_customr_wkly_tbl'
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE

tx_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_tbl_final_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

sob_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_SOB_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
episode_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_EPISODE_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
regimen_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_BASIC_REGIMEN_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

patient_group = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LoT IC INELIGIBLE
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LOT IC ELIGIBLE
ic_elig_final = '{cws}.MABI_AML_IC_ELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws) # LOT table IC ELIG
pat_comb_lot = '{cws}.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)  # UNION OF BOTH LOT FINAL TABLES
pat_comb_lot_rr = '{cws}.MABI_AML_PATS_LOT_RR_12M_UPD_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)    

pat_fst_dx_tx = '{cws}.MABI_AML_LOT_PAT_FST_DX_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

dx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_DX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx = '{cws}.MABI_AML_LOT_PAT_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig = '{cws}.MABI_AML_LOT_PAT_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

tx_lookback_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

lot_regimen_group = '{cws}.MABI_AML_PATS_LOT_GROUPING_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
patient_intensity_based_on_age_n_product = '{cws}.PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT_v2'.format(cws=cws)

#HCP AFFILIATIONS
hcp_affiliations = '{cws}.MABI_AML_HCP_AFFILIATIONS'.format(cws=cws)

# LoT TABLE ELIGIBILITY CHECK
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

#RR AML DX ONLY
aml_dx = ('C92.01', 'C92.02', 'C92.51', 'C92.52', 'C92.61', 'C92.62', 'C92.A1', 'C92.A2', 'C93.01', 'C93.02', 'C94.01', 'C94.21', 'C94.22')

#Product codes for Onureg, Inqovi, Rezlidhia and Vanflyta 

onureg_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%AZACITIDINE%') 
          AND SOURCE_DRUG_FORM IN ('TAB','CAP')
                                        
                                        UNION ALL
                                 select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(PROCEDURE_DESCription) LIKE ('%ONUREG%')
                                        
                                 ''') 
df1 = onureg_product_code.toPandas()
df2 = df1['PRODUCT_NDC_CODE'].values.tolist()
df3 = list(df2)
onureg_codes = str(df3)[1:-1]

inqovi_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%DECITABINE/CEDAZURIDINE%') 
          AND SOURCE_DRUG_FORM IN ('TAB','CAP')        
                                 
                                       
                                        UNION ALL
                                        select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%INQOVI%') or UPPER(PROCEDURE_DESCription) LIKE ('%DECITABINE/CEDAZURIDINE%'))
                                 
                                       
                                ''') 
df4 = inqovi_product_code.toPandas()
df5 = df4['PRODUCT_NDC_CODE'].values.tolist()
df6 = list(df5)
inqovi_codes = str(df6)[1:-1]

rezlidhia_product_code =  spark.sql('''
                                        select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%OLUTASIDENIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%REZLIDHIA%'))

                                        
                                        UNION ALL
                                    select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%REZLIDHIA%') or UPPER(PROCEDURE_DESCription) LIKE ('%OLUTASIDENIB%'))


                                ''') 
df7 = rezlidhia_product_code.toPandas()
df8 = df7['PRODUCT_NDC_CODE'].values.tolist()
df9 = list(df8)
rezlidhia_codes = str(df9)[1:-1]

vanflyta_product_code =  spark.sql('''  
                                   select distinct source_product_ndc_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.dim_product_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%QUIZARTINIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%VANFLYTA%'))


                                         
                                        UNION ALL
                                   select distinct procedure_code as PRODUCT_NDC_CODE
          from abv_val_ptd_onc_synd.px_fact_curated_vw
          where source_market = 'ONC'
          and source_flag = 'SHA_PTD'
          and (UPPER(PROCEDURE_DESCription) LIKE ('%VANFLYTA%') or UPPER(PROCEDURE_DESCription) LIKE ('%QUIZARTINIB%'))


                                ''') 
df10 = vanflyta_product_code.toPandas()
df11 = df10['PRODUCT_NDC_CODE'].values.tolist()
df12 = list(df11)
vanflyta_codes = str(df12)[1:-1]


#### New Patient Pool of AML 2+ Dx

New Code

In [ ]:
# New Patient Pool Created

patient_pool = spark.sql('''
SELECT distinct PATIENT_SK from
(
    select patient_sk, count(distinct mx_claim_id) as Count_of_claims
    FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
    WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
    (
        '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
        'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
        'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
        'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
    )
    and source_flag='SHA_PTD'
    group by 1
)
where count_of_claims>1
          ''')

patient_pool.createOrReplaceTempView('patient_pool')

# TX TABLE UPDATE

#### Creating the Tx Table

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_tbl_name} '''.format(tx_tbl_name=tx_tbl_name))

Changes Made

In [ ]:
# Removed the Patient Indication Filter (Diagnosis Frequency, Recency, and Hierarchy criteria) and instead identify patients based on AML 2+ diagnosis.

spark.sql('''
CREATE TABLE {tx_tbl_name} AS
(
          select distinct patient_sk as patient_gid, PRODUCT_QTY_DISPENSED, PRODUCT_DAYS_SUPPLY, 
          mastered_product_name as final_product_name, claim_date, claim_id, MDM_NPI_NUMBER as npi_number, PROCEDURE_CODE as product_code, source_type as native_type
          FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
            where source_flag = 'SHA_PTD'
            and source_market = 'ONC'
            --and market_code = 'VENC_AML'   
            and patient_sk in (
                select patient_sk
                from patient_pool
            )
            and mastered_product_name IN (
                'AZACITIDINE','DAURISMO','DECITABINE','IDHIFA','L-DAC','TIBSOVO','VENCLEXTA',
                'XOSPATA','REZLIDHIA','MYLOTARG','RYDAPT','ARSENIC_TRIOXIDE','CLADRIBINE','CLOFARABINE','CYCLOPHOSPHAMIDE',
                'DAUNORUBICIN','ETOPOSIDE','FLUDARABINE','HI-DAC','IDARUBICIN','MITOXANTRONE','NEXAVAR','ONUREG','INQOVI',
                'S-DAC','VINCRISTINE','VANFLYTA','VYXEOS'
            )

)
'''.format(tx_tbl_name=tx_tbl_name))




In [ ]:
spark.sql(''' 
select count(distinct patient_gid) from
{tx_tbl_name}

'''.format(tx_tbl_name=tx_tbl_name)).show()

#### DOS AND GRACE ASSIGNMENT

In [ ]:
tx_tbl_final_name

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {tx_tbl_final_name}'''.format(tx_tbl_final_name=tx_tbl_final_name))

#### DOS AND GRACE CONFIGURATION

DOS and GRACE values per product are held in
`../configuration/aml_dos_grace.yaml`, not in the SQL below. The helper builds the
`GRACE_VALUE` and `DOS_FINAL` CASE expressions from that file.

To change a value use the `/update-dos-grace` skill, or edit the YAML and run
`python validate_dos_grace.py`. A DOS or GRACE change moves reported lines of
therapy — record the reason in `CONTEXT.md`.

In [ ]:
# Build the DOS/GRACE CASE expressions from configuration.
# Path resolves relative to this notebook; override with AML_DOS_GRACE_CONFIG.
from dos_grace_config import (
    load_config,
    build_grace_case_sql,
    build_dos_case_sql,
    build_product_exclusion_sql,
    summarise,
)

dos_grace_cfg = load_config()

grace_case_sql       = build_grace_case_sql(dos_grace_cfg)
dos_case_sql         = build_dos_case_sql(dos_grace_cfg)
product_exclusion_sql = build_product_exclusion_sql(dos_grace_cfg)

# Confirm what will be applied before the table is written.
print(summarise(dos_grace_cfg))

In [ ]:
spark.sql('''
CREATE TABLE {tx_tbl_final_name} AS
(
    SELECT *
    FROM
    (
        SELECT *,
        /* Grace period by product - from configuration/aml_dos_grace.yaml */
        {grace_case_sql} AS GRACE_VALUE
        FROM
        (
            SELECT *,
            /* Days of supply: claim value for RX, per-product value for PX */
            {dos_case_sql} AS DOS_FINAL
            FROM
            (
                SELECT DISTINCT PATIENT_GID, PRODUCT_QTY_DISPENSED, NPI_NUMBER,
                CASE
                    WHEN NATIVE_TYPE LIKE '%RX%' THEN 'RX'
                    ELSE 'PX'
                END AS NATIVE_TYPE,
                CLAIM_ID, CLAIM_DATE, PRODUCT_CODE, FINAL_PRODUCT_NAME AS FINAL_PRODUCT_NAME, 
                PRODUCT_DAYS_SUPPLY AS PRODUCT_DAYS_SUPPLY
                FROM {tx_tbl_name}
                /* Excluded products - from configuration/aml_dos_grace.yaml */
                WHERE {product_exclusion_sql}
            )A
        )A
    )A
)    

'''.format(
    tx_tbl_name=tx_tbl_name,
    tx_tbl_final_name=tx_tbl_final_name,
    grace_case_sql=grace_case_sql,
    dos_case_sql=dos_case_sql,
    product_exclusion_sql=product_exclusion_sql,
))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {tx_tbl_final_name}
'''.format(tx_tbl_final_name=tx_tbl_final_name)).toPandas().head(20)

# REGIMEN CREATION

#### SOB

In [ ]:
sob_tbl_name

In [ ]:
sob_df = stencil.init(spark).sob(
    grace='GRACE_VALUE'
	, lookback='360'
	, dedup_type_vl='yesremove' #to handle multiple claims for a product on same day
	, dedup_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, CLAIM_DATE'
	, dedup_ranking_order_clmns='DOS_FINAL DESC, CLAIM_ID' #choose claim with max DOS among multiple claims for a product on same day
	, dos_clmn='DOS_FINAL'
	, patient_gid_clmn='PATIENT_GID'
	, claim_id_clmn='CLAIM_ID'
	, claim_start_dt_clmn='CLAIM_DATE'
	, product_name_clmn='FINAL_PRODUCT_NAME'
	, src_input_tbl='(SELECT *, "202210" AS DATA_PERIOD FROM {tx_tbl_final_name})'.format(tx_tbl_final_name=tx_tbl_final_name)
	, metric_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, metric_ordering_by_following_clmns='CLAIM_DATE, CLAIM_ID'
	, dos_metric_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, CLAIM_DATE'
	, data_end_required='no'
	, required_grace='0'
)

sob_df.createOrReplaceTempView('sob_df')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {sob_tbl_name}'''.format(sob_tbl_name=sob_tbl_name))

In [ ]:
spark.sql('''
CREATE TABLE {sob_tbl_name} AS
(
    SELECT *
    FROM sob_df
)
'''.format(sob_tbl_name=sob_tbl_name))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {sob_tbl_name}
'''.format(sob_tbl_name=sob_tbl_name)).toPandas().head(20)

#### EPISODE

In [ ]:
episode_tbl_name

In [ ]:
episode_df = stencil.init(spark).episode(
	src_input_tbl='sob_df'
	, patient_gid_clmn='PATIENT_GID'
	, product_name_clmn='FINAL_PRODUCT_NAME'
	, metric_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, claim_rnkn_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, EPISODE_NUM_DRVD'
	, claim_rnkn_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, claim_id_clmn='CLAIM_ID'
	, episode_end_dts_clmns='EPISODE_END_DATE1_DRVD, EPISODE_END_DATE2_DRVD, EPISODE_END_DATE3_DRVD'
	, claim_start_dt_clmn='CLAIM_DATE'
	, drug_class_clmn ='"NA"'
	, dos_null_flag_clmns='DOS_NULL_FLAG, DOS_PAT_PRD_NULL_FLAG, DOS_PAT_PRD_EPSD_NULL_FLAG'
)

episode_df.createOrReplaceTempView('episode_df')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {episode_tbl_name}'''.format(episode_tbl_name=episode_tbl_name))

In [ ]:
spark.sql('''
CREATE TABLE {episode_tbl_name} AS
(
    SELECT *
    FROM episode_df
)
'''.format(episode_tbl_name=episode_tbl_name))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {episode_tbl_name}
'''.format(episode_tbl_name=episode_tbl_name)).toPandas().head(20)

#### REGIMEN

In [ ]:
regimen_tbl_name

In [ ]:
regimen_df = stencil.init(spark).regimen(
	src_input_tbl='episode_df'
	, patient_gid_clmn='PATIENT_GID'
	, product_name_clmn='FINAL_PRODUCT_NAME'
	, episode_end_dt_clmn='EPISODE_END_DATE1_DRVD'
	, final_indication='"TEST"'
	, regimen_threshold_vl='5000'
	, clean_up_type_vl='no'
	, regimen_removal_vl='0,1'
	, episode_start_dt_clmn='EPISODE_START_DATE_DRVD'
	, collect_type_rgmn_vl='COLLECT_SET'
	, collect_type_drg_rgmn_vl='COLLECT_SET'
	, data_end_dt_vl='2400-01-01'
	, drug_class_clmn='"test"'
	, regimen_replace_clmn='regimen'
	, product_replace_from_vls='"TEST"'
	, product_replace_to_vls='"TEST"'
)

regimen_df.createOrReplaceTempView('regimen_df')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {regimen_tbl_name}'''.format(regimen_tbl_name=regimen_tbl_name))

In [ ]:
spark.sql('''
CREATE TABLE {regimen_tbl_name} AS
(
    SELECT *
    FROM regimen_df
)
'''.format(regimen_tbl_name=regimen_tbl_name))

In [ ]:
spark.sql('''SELECT COUNT(DISTINCT PATIENT_GID_RGMN) FROM {regimen_tbl_name}'''.format(regimen_tbl_name=regimen_tbl_name)).show()

# PATIENT COHORT

In [ ]:
patient_group

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {patient_group}'''.format(patient_group=patient_group))

Changes Made

In [ ]:
# INQOVI and ONUREG are classified as IC Ineligible products.
# Rydapt and Mylotarg are classified as IC Eligible only.

spark.sql('''
CREATE TABLE {patient_group} AS
(
    SELECT A.*,
           CASE WHEN PATIENT_INTENSITY = 'IC_INELIG' THEN 'IC_INELIG'
           ELSE 'IC_ELIG'
           END AS PATIENT_INTENSITY_GROUP
    FROM
    (
        SELECT *
        FROM
        (
            SELECT *,
            CASE
                WHEN LOW_INTENSITY = 1 THEN 'IC_INELIG'
                ELSE 'IC_ELIG'
            END AS PATIENT_INTENSITY
            FROM
            (
                SELECT PATIENT_GID_RGMN, MIN(LOW_INTENSITY_INT) AS LOW_INTENSITY
                FROM
                (

                    --Simplified the code as well
          
                    SELECT *, LOW_INTENSITY AS LOW_INTENSITY_INT
                    
                    
                    --SELECT *,
                    --CASE    
                    --    WHEN LOW_INTENSITY = 0 THEN 0
                    --    WHEN LOW_INTENSITY = 0 AND RYD_MYL_FLG = 1 THEN 1
                    --    ELSE LOW_INTENSITY
                    --END AS LOW_INTENSITY_INT


                    FROM
                    (
                        SELECT *,
                        CASE 
          
                            WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 0
                            WHEN REGIMEN LIKE '%CLADRIBINE%' THEN 0
                            WHEN REGIMEN LIKE '%CLOFARABINE%' THEN 0
                            WHEN REGIMEN LIKE '%CYCLOPHOSPHAMIDE%' THEN 0
                            WHEN REGIMEN LIKE '%S-DAC%' THEN 0
                            WHEN REGIMEN LIKE '%HI-DAC%' THEN 0
                            WHEN REGIMEN LIKE '%DAUNORUBICIN%' THEN 0
                            WHEN REGIMEN LIKE '%ETOPOSIDE%' THEN 0
                            WHEN REGIMEN LIKE '%FILGRASTIM%' THEN 0
                            WHEN REGIMEN LIKE '%FLUDARABINE%' THEN 0
                            WHEN REGIMEN LIKE '%IDARUBICIN%' THEN 0
                            --WHEN REGIMEN LIKE '%INQOVI%' THEN 0  --NOW INQOVI CONSIDERED AS IC INELIG
                            WHEN REGIMEN LIKE '%MITOXANTRONE%' THEN 0
                            WHEN REGIMEN LIKE '%MYLOTARG%' THEN 0
                            WHEN REGIMEN LIKE '%NEXAVAR%' THEN 0
                            --WHEN REGIMEN LIKE '%ONUREG%' THEN 0  --NOW ONUREG CONSIDERED AS IC INELIG
                            WHEN REGIMEN LIKE '%RYDAPT%' THEN 0
                            WHEN REGIMEN LIKE '%VANFLYTA%' THEN 0
                            WHEN REGIMEN LIKE '%VINCRISTINE%' THEN 0
                            WHEN REGIMEN LIKE '%VYXEOS%' THEN 0
                            WHEN REGIMEN LIKE '%RYDAPT%' THEN 0
                            WHEN REGIMEN LIKE '%MYLOTARG%' THEN 0
                            ELSE 1
                        END AS LOW_INTENSITY

                        --Special condition for Rydapt and Mylotarg removed since they are considered only as IC Elig
                        --,CASE 
                        --    WHEN REGIMEN = ('RYDAPT, VENCLEXTA') OR REGIMEN = ('MYLOTARG, VENCLEXTA') 
                        --    OR REGIMEN = ('AZACITIDINE, RYDAPT') OR REGIMEN = ('DECITABINE, RYDAPT') 
                        --    OR REGIMEN = ('DECITABINE, MYLOTARG') OR REGIMEN = ('AZACITIDINE, MYLOTARG') THEN 1 
                        --    ELSE 0
                        --END AS RYD_MYL_FLG

                        FROM 
                        (
                            SELECT DISTINCT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE
                            FROM {regimen_tbl_name}
                        )A
                    )A
                )A
                GROUP BY 1
            )A
        )A
    )A
)
'''.format(patient_group=patient_group,regimen_tbl_name=regimen_tbl_name,pat_fst_dx_tx=pat_fst_dx_tx))

In [ ]:
spark.sql('''
SELECT PATIENT_INTENSITY_GROUP, COUNT(DISTINCT PATIENT_GID_RGMN) AS NUM_PAT
FROM {patient_group}
GROUP BY 1
'''.format(patient_group=patient_group)).toPandas().head(20)

In [ ]:
spark.sql('''
SELECT DISTINCT FINAL_PRODUCT_NAME
FROM {tx_tbl_final_name}
WHERE PATIENT_GID IN
(
  SELECT DISTINCT PATIENT_GID_RGMN
  FROM {patient_group}
  WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
)
'''.format(tx_tbl_final_name=tx_tbl_final_name,patient_group=patient_group)).toPandas().head(20)

# REGIMEN CONSOLIDATION - IC INELIG

#### PREVIOUS MAJOR

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_inelig_cons1 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE
# 				WHEN MAJOR_REGIMEN = 1 AND PREV_MAJOR_REGIMEN = 1 AND REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_FROM_PREV = 0 THEN 0
# 				ELSE 1
# 			END AS CONSOL
# 			FROM
# 			(	
# 				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, MAJOR_REGIMEN,
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END, PREV_MAJOR_REGIMEN,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 				FROM
# 				(
# 					SELECT *,
# 					CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 					FROM
# 					(
# 						SELECT *,
# 						CASE WHEN PREV_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 						FROM
# 						(
# 							SELECT *,
# 							DATEDIFF(TO_DATE(REGIMEN_START_DATE), TO_DATE(PREV_REGIMEN_END)) AS GAP_FROM_PREV
# 							FROM
# 							(
# 								SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, MAJOR_REGIMEN,
# 								LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 								LAG(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN,
# 								LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN_END,
# 								LAG(MAJOR_REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_MAJOR_REGIMEN
# 								FROM
# 								(
# 									SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH,
# 									CASE
# 										WHEN REGIMEN = 'VENCLEXTA' THEN 1
# 										WHEN REGIMEN = 'TIBSOVO' THEN 1
# 										WHEN REGIMEN = 'REZLIDHIA' THEN 1
# 										WHEN REGIMEN = 'IDHIFA' THEN 1
# 										WHEN REGIMEN = 'XOSPATA' THEN 1
# 										WHEN REGIMEN = 'DAURISMO' THEN 1
# 										WHEN REGIMEN = 'L-DAC' THEN 1
# 										WHEN REGIMEN = 'AZACITIDINE' THEN 1
# 										WHEN REGIMEN = 'DECITABINE' THEN 1
# 										WHEN REGIMEN = 'AZACITIDINE, VENCLEXTA' THEN 1
# 										WHEN REGIMEN = 'DECITABINE, VENCLEXTA' THEN 1
# 										WHEN REGIMEN = 'RYDAPT, VENCLEXTA' THEN 1
# 										WHEN REGIMEN = 'MYLOTARG, VENCLEXTA' THEN 1
# 										WHEN REGIMEN = 'AZACITIDINE, RYDAPT' THEN 1
# 										WHEN REGIMEN = 'AZACITIDINE, MYLOTARG' THEN 1
# 										WHEN REGIMEN = 'DECITABINE, RYDAPT' THEN 1
# 										WHEN REGIMEN = 'DECITABINE, MYLOTARG' THEN 1
# 										ELSE 0
# 									END AS MAJOR_REGIMEN
# 									FROM {regimen_tbl_name}
# 									WHERE PATIENT_GID_RGMN IN
# 									(
# 										SELECT DISTINCT PATIENT_GID_RGMN
# 										FROM {patient_group}
# 										WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
# 									)
# 								)A
# 							)A
# 						)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 					)A
# 				)A
# 				GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, MAJOR_REGIMEN,
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END, PREV_MAJOR_REGIMEN,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# '''.format(regimen_tbl_name=regimen_tbl_name,patient_group=patient_group))

# ic_inelig_cons1.createOrReplaceTempView('ic_inelig_cons1')

#### NEXT - MAJOR

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_inelig_cons2 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE 
# 				WHEN PREV_MERGE_TO_NEXT = 1 THEN 0
# 				ELSE 1
# 			END AS CONSOL 
# 			FROM
# 			(
# 				SELECT *,
# 				LAG(MERGE_TO_NEXT) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_MERGE_TO_NEXT
# 				FROM
# 				(
# 					SELECT *,
# 					CASE 
# 						WHEN MAJOR_REGIMEN = 1 AND NEXT_MAJOR_REGIMEN = 1 AND REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_TO_NEXT = 0 THEN 1
# 						ELSE 0
# 					END AS MERGE_TO_NEXT
# 					FROM
# 					(
# 						SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, MAJOR_REGIMEN,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START, NEXT_MAJOR_REGIMEN,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 						FROM
# 						(
# 							SELECT *,
# 							CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 							FROM
# 							(
# 								SELECT *,
# 								CASE WHEN NEXT_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 								FROM
# 								(
# 									SELECT *,
# 									DATEDIFF(TO_DATE(REGIMEN_END_DATE), TO_DATE(REGIMEN_START_DATE)) AS REGIMEN_LENGTH,
# 									DATEDIFF(TO_DATE(NEXT_REGIMEN_START), TO_DATE(REGIMEN_END_DATE)) AS GAP_TO_NEXT
# 									FROM
# 									(
# 										SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, MAJOR_REGIMEN,
# 										LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 										LEAD(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN,
# 										LEAD(REGIMEN_START_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN_START,
# 										LEAD(MAJOR_REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_MAJOR_REGIMEN
# 										FROM 
# 										(
# 											SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 											CASE
# 												WHEN REGIMEN = 'VENCLEXTA' THEN 1
# 												WHEN REGIMEN = 'TIBSOVO' THEN 1
# 												WHEN REGIMEN = 'REZLIDHIA' THEN 1
# 												WHEN REGIMEN = 'IDHIFA' THEN 1
# 												WHEN REGIMEN = 'XOSPATA' THEN 1
# 												WHEN REGIMEN = 'DAURISMO' THEN 1
# 												WHEN REGIMEN = 'L-DAC' THEN 1
# 												WHEN REGIMEN = 'AZACITIDINE' THEN 1
# 												WHEN REGIMEN = 'DECITABINE' THEN 1
# 												WHEN REGIMEN = 'AZACITIDINE, VENCLEXTA' THEN 1
# 												WHEN REGIMEN = 'DECITABINE, VENCLEXTA' THEN 1
# 												WHEN REGIMEN = 'RYDAPT, VENCLEXTA' THEN 1
# 												WHEN REGIMEN = 'MYLOTARG, VENCLEXTA' THEN 1
# 												WHEN REGIMEN = 'AZACITIDINE, RYDAPT' THEN 1
# 												WHEN REGIMEN = 'AZACITIDINE, MYLOTARG' THEN 1
# 												WHEN REGIMEN = 'DECITABINE, RYDAPT' THEN 1
# 												WHEN REGIMEN = 'DECITABINE, MYLOTARG' THEN 1
# 												ELSE 0
# 											END AS MAJOR_REGIMEN
# 											FROM ic_inelig_cons1
# 										)
# 									)A
# 								)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 							)A
# 						)A
# 						GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, MAJOR_REGIMEN,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START, NEXT_MAJOR_REGIMEN,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 					)A
# 				)A
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# ''')

# ic_inelig_cons2.createOrReplaceTempView('ic_inelig_cons2')

#### PREVIOUS

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_inelig_cons3 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE
# 				WHEN REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_FROM_PREV = 0 THEN 0
# 				ELSE 1
# 			END AS CONSOL
# 			FROM
# 			(	
# 				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, 
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 				FROM
# 				(
# 					SELECT *,
# 					CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 					FROM
# 					(
# 						SELECT *,
# 						CASE WHEN PREV_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 						FROM
# 						(
# 							SELECT *,
# 							DATEDIFF(TO_DATE(REGIMEN_START_DATE), TO_DATE(PREV_REGIMEN_END)) AS GAP_FROM_PREV
# 							FROM
# 							(
# 								SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH,
# 								LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 								LAG(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN,
# 								LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN_END
# 								FROM
# 								(
# 									SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, 
# 									DATEDIFF(TO_DATE(REGIMEN_END_DATE), TO_DATE(REGIMEN_START_DATE)) AS REGIMEN_LENGTH
# 									FROM ic_inelig_cons2
# 								)A
# 							)A
# 						)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 					)A
# 				)A
# 				GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, 
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# ''')

# ic_inelig_cons3.createOrReplaceTempView('ic_inelig_cons3')

#### NEXT

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_inelig_cons4 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE 
# 				WHEN PREV_MERGE_TO_NEXT = 1 THEN 0
# 				ELSE 1
# 			END AS CONSOL 
# 			FROM
# 			(
# 				SELECT *,
# 				LAG(MERGE_TO_NEXT) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_MERGE_TO_NEXT
# 				FROM
# 				(
# 					SELECT *,
# 					CASE 
# 						WHEN REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_TO_NEXT = 0 THEN 1
# 						ELSE 0
# 					END AS MERGE_TO_NEXT
# 					FROM
# 					(
# 						SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 						FROM
# 						(
# 							SELECT *,
# 							CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 							FROM
# 							(
# 								SELECT *,
# 								CASE WHEN NEXT_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 								FROM
# 								(
# 									SELECT *,
# 									DATEDIFF(TO_DATE(REGIMEN_END_DATE), TO_DATE(REGIMEN_START_DATE)) AS REGIMEN_LENGTH,
# 									DATEDIFF(TO_DATE(NEXT_REGIMEN_START), TO_DATE(REGIMEN_END_DATE)) AS GAP_TO_NEXT
# 									FROM
# 									(
# 										SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 										LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 										LEAD(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN,
# 										LEAD(REGIMEN_START_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN_START
# 										FROM 
# 										(
# 											SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE
# 											FROM ic_inelig_cons3
# 										)
# 									)A
# 								)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 							)A
# 						)A
# 						GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 					)A
# 				)A
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# ''')

# ic_inelig_cons4.createOrReplaceTempView('ic_inelig_cons4')

#### FINAL CONS - LOW INTENSITY + VALID LENGTH

Changes Made

In [ ]:
# Removing exception condition for Rydapt and smaller regimen dropping condition

ic_inelig_cons_final = spark.sql('''
SELECT *
FROM
(
    -- Smaller regimens were dropped earlier, now all the regimens are kept as it is
                                 
    SELECT *  
    --,CASE
      --  WHEN REGIMEN = 'RYDAPT' AND REGIMEN_LENGTH >= 14 THEN 1 --RYDAPT CLAIMS HAVE DOS OF 14/21 AS WELL
      --  WHEN REGIMEN_LENGTH >= 28 THEN 1
     --   ELSE 0
    --END AS SEL_FLAG
                                 
    FROM
    (   SELECT PATIENT_GID_RGMN AS PATIENT_GID,  REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
        --SELECT PATIENT_GID_RGMN AS PATIENT_GID, NEW_RNK, REGIMEN, LVL, REGIMEN_START_DATE, REGIMEN_END_DATE,
        DATEDIFF(REGIMEN_END_DATE, REGIMEN_START_DATE) AS REGIMEN_LENGTH
        --FROM ic_inelig_cons5
         FROM {regimen_tbl_name}
 									WHERE PATIENT_GID_RGMN IN
 									(
 										SELECT DISTINCT PATIENT_GID_RGMN
 										FROM {patient_group}
 										WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
 									)
    )A
)A
                                 
--WHERE SEL_FLAG = 1
                                 
'''.format(regimen_tbl_name=regimen_tbl_name, patient_group=patient_group))

ic_inelig_cons_final.createOrReplaceTempView('ic_inelig_cons_final')

#### LoT ASSIGNMENT

In [ ]:
ic_inelig_final

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {ic_inelig_final}'''.format(ic_inelig_final=ic_inelig_final))

Changes Made

In [ ]:
# Addition of Inqovi and Onureg as HMAs and removal of Rydapt and Mylotarg in Backbone Product hierarchy

spark.sql('''
CREATE TABLE {ic_inelig_final} AS
(	
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 --WHEN BACKBONE <> PREV_BACKBONE THEN 1
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF(REGIMEN_START_DATE, LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID, REGIMEN, 
				TO_DATE(REGIMEN_START_DATE) AS REGIMEN_START_DATE, 
				TO_DATE(REGIMEN_END_DATE) AS REGIMEN_END_DATE, REGIMEN_LENGTH,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
          
					WHEN REGIMEN RLIKE 'VENCLEXTA' THEN 'VENCLEXTA'
					WHEN REGIMEN RLIKE 'TIBSOVO' THEN 'TIBSOVO'
					WHEN REGIMEN RLIKE 'REZLIDHIA' THEN 'REZLIDHIA'
					WHEN REGIMEN RLIKE 'IDHIFA' THEN 'IDHIFA'
					WHEN REGIMEN RLIKE 'XOSPATA' THEN 'XOSPATA'
					WHEN REGIMEN RLIKE 'DAURISMO' THEN 'DAURISMO'
					--WHEN REGIMEN RLIKE 'RYDAPT' THEN 'RYDAPT'   --REMOVING RYDAPT AND MYLOTARG FROM IC INELIG BACKBONE HIERARCHY
					--WHEN REGIMEN RLIKE 'MYLOTARG' THEN 'MYLOTARG'
					WHEN REGIMEN RLIKE 'L-DAC' THEN 'L-DAC'
					WHEN REGIMEN RLIKE 'AZACITIDINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'DECITABINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'INQOVI' THEN 'HMA'		--ADDING INQOVI AND ONUREG IN IC INELIG BACKBONE HIERARCHY
					WHEN REGIMEN RLIKE 'ONUREG' THEN 'HMA'
				END AS BACKBONE
				FROM ic_inelig_cons_final
			)A
		)A
	)
)
'''.format(ic_inelig_final=ic_inelig_final))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {ic_inelig_final}
'''.format(ic_inelig_final=ic_inelig_final)).show()

# REGIMEN CONSOLIDATION - IC ELIG

#### PREVIOUS

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation


# ic_elig_cons1 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE
# 				WHEN REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_FROM_PREV = 0 THEN 0
# 				ELSE 1
# 			END AS CONSOL
# 			FROM
# 			(	
# 				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, 
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 				FROM
# 				(
# 					SELECT *,
# 					CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 					FROM
# 					(
# 						SELECT *,
# 						CASE WHEN PREV_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 						FROM
# 						(
# 							SELECT *,
# 							DATEDIFF(TO_DATE(REGIMEN_START_DATE), TO_DATE(PREV_REGIMEN_END)) AS GAP_FROM_PREV
# 							FROM
# 							(
# 								SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH,
# 								LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 								LAG(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN,
# 								LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN_END
# 								FROM
# 								(
# 									SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, 
# 									DATEDIFF(TO_DATE(REGIMEN_END_DATE), TO_DATE(REGIMEN_START_DATE)) AS REGIMEN_LENGTH
# 									FROM {regimen_tbl_name}
# 									WHERE PATIENT_GID_RGMN IN
# 									(
# 										SELECT DISTINCT PATIENT_GID_RGMN
# 										FROM {patient_group}
# 										WHERE PATIENT_INTENSITY_GROUP = 'IC_ELIG'
# 									)
# 								)A
# 							)A
# 						)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 					)A
# 				)A
# 				GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, REGIMEN_LENGTH, 
# 				LVL, PREV_REGIMEN, PREV_REGIMEN_END,
# 				GAP_FROM_PREV, SUBSET_FLAG
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# '''.format(regimen_tbl_name=regimen_tbl_name,patient_group=patient_group))

# ic_elig_cons1.createOrReplaceTempView('ic_elig_cons1')

In [ ]:
# spark.sql('''
# SELECT COUNT(DISTINCT PATIENT_GID_RGMN)
# FROM ic_elig_cons1
# ''').show()

#### NEXT

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_elig_cons2 = spark.sql('''
# SELECT PATIENT_GID_RGMN, NEW_RNK,
# MAX(NEW_REGIMEN) AS REGIMEN, MAX(LVL) AS LVL, MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE
# FROM
# (
# 	SELECT *,
# 	CASE 
# 		WHEN LVL = (MAX(LVL) OVER (PARTITION BY PATIENT_GID_RGMN, NEW_RNK ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)) THEN REGIMEN 
# 		ELSE NULL 
# 	END AS NEW_REGIMEN 
# 	FROM
# 	(
# 		SELECT *,
# 		SUM(CONSOL) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
# 		FROM
# 		(
# 			SELECT *,
# 			CASE 
# 				WHEN PREV_MERGE_TO_NEXT = 1 THEN 0
# 				ELSE 1
# 			END AS CONSOL 
# 			FROM
# 			(
# 				SELECT *,
# 				LAG(MERGE_TO_NEXT) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_MERGE_TO_NEXT
# 				FROM
# 				(
# 					SELECT *,
# 					CASE 
# 						WHEN REGIMEN_LENGTH < 28 AND SUBSET_FLAG = 1 AND GAP_TO_NEXT = 0 THEN 1
# 						ELSE 0
# 					END AS MERGE_TO_NEXT
# 					FROM
# 					(
# 						SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 						FROM
# 						(
# 							SELECT *,
# 							CASE WHEN LVL = SUM(RGMN_CHK) OVER (PARTITION BY PATIENT_GID_RGMN, REGIMEN_START_DATE) THEN 1 ELSE 0 END AS SUBSET_FLAG 
# 							FROM
# 							(
# 								SELECT *,
# 								CASE WHEN NEXT_REGIMEN RLIKE REGIMEN_PRODUCTS THEN 1 ELSE 0 END AS RGMN_CHK
# 								FROM
# 								(
# 									SELECT *,
# 									DATEDIFF(TO_DATE(REGIMEN_END_DATE), TO_DATE(REGIMEN_START_DATE)) AS REGIMEN_LENGTH,
# 									DATEDIFF(TO_DATE(NEXT_REGIMEN_START), TO_DATE(REGIMEN_END_DATE)) AS GAP_TO_NEXT
# 									FROM
# 									(
# 										SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 										LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
# 										LEAD(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN,
# 										LEAD(REGIMEN_START_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS NEXT_REGIMEN_START
# 										FROM 
# 										(
# 											SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE
# 											FROM ic_elig_cons1
# 										)
# 									)A
# 								)AS REGIMEN_FLAG LATERAL VIEW EXPLODE(SPLIT(REGIMEN,', ')) REGIMEN AS REGIMEN_PRODUCTS
# 							)A
# 						)A
# 						GROUP BY PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
# 						LVL, NEXT_REGIMEN, NEXT_REGIMEN_START,
# 						REGIMEN_LENGTH, GAP_TO_NEXT, SUBSET_FLAG
# 					)A
# 				)A
# 			)A
# 		)A
# 	)A
# )A
# GROUP BY PATIENT_GID_RGMN, NEW_RNK
# ''')

# ic_elig_cons2.createOrReplaceTempView('ic_elig_cons2')

Code to be Removed

In [ ]:
# Removing Regimen Consolidation rule before LoT formation

# ic_elig_cons3 = spark.sql( '''
#         SELECT PATIENT_GID_RGMN, NEW_RNK, REGIMEN,
#         MIN(REGIMEN_START_DATE) AS REGIMEN_START_DATE, MAX(REGIMEN_END_DATE) AS REGIMEN_END_DATE, MAX(LVL) AS LVL
#         FROM
#         (
#             SELECT *,
#             SUM(FLAG) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS NEW_RNK
#             FROM
#             (
#                 SELECT *,
#                 CASE
#                     WHEN GAP_FROM_PREV = 0 AND REGIMEN = PREV_REGIMEN THEN 0
#                     ELSE 1
#                 END AS FLAG
#                 FROM
#                 (
#                     SELECT *,
#                     DATEDIFF(TO_DATE(REGIMEN_START_DATE), TO_DATE(PREV_REGIMEN_END)) AS GAP_FROM_PREV,
#                     LAG(REGIMEN) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN
#                     FROM
#                     (
#                         SELECT DISTINCT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
#                         LENGTH(REGEXP_REPLACE(REGIMEN,'[^,]',''))+1 AS LVL,
#                         LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_REGIMEN_END
#                         FROM ic_elig_cons2
#                     )
#                 )
#             )
#         )
#         GROUP BY 1,2,3
# ''')

# ic_elig_cons3.createOrReplaceTempView('ic_elig_cons3')

#### LOT ASSIGNMENT - IC ELIG

In [ ]:
ic_elig_final

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {ic_elig_final}'''.format(ic_elig_final=ic_elig_final))

Changes Made

In [ ]:
# Inqovi and ONUREG are treated as HMAs

spark.sql('''
CREATE TABLE {ic_elig_final} AS
( 
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0	
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 											
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF((REGIMEN_START_DATE), LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, 
				DATEDIFF((REGIMEN_END_DATE) , (REGIMEN_START_DATE)) AS REG_LEN,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
					WHEN REGIMEN LIKE '%HI-DAC%' OR REGIMEN LIKE '%S-DAC%' THEN 'S-DAC/HI-DAC'
					WHEN REGIMEN LIKE '%VANFLYTA%' THEN 'VANFLYTA'
					WHEN REGIMEN LIKE '%VYXEOS%' THEN 'VYXEOS'
					WHEN REGIMEN LIKE '%CYCLOPHOSPHAMIDE%' THEN 'CYCLOPHOSPHAMIDE'
					WHEN REGIMEN LIKE '%VINCRISTINE%' THEN 'VINCRISTINE'
					WHEN REGIMEN LIKE '%FLUDARABINE%' THEN 'FLUDARABINE'
					WHEN REGIMEN LIKE '%IDARUBICIN%' THEN 'IDARUBICIN'
					WHEN REGIMEN LIKE '%DAUNORUBICIN%' THEN 'DAUNORUBICIN'
					WHEN REGIMEN LIKE '%MITOXANTRONE%' THEN 'MITOXANTRONE'
					WHEN REGIMEN LIKE '%RYDAPT%' THEN 'RYDAPT'
					WHEN REGIMEN LIKE '%VENCLEXTA%' THEN 'VENCLEXTA'
					WHEN REGIMEN LIKE '%XOSPATA%' THEN 'XOSPATA'
					WHEN REGIMEN LIKE '%TIBSOVO%' THEN 'TIBSOVO'
					WHEN REGIMEN LIKE '%REZLIDHIA%' THEN 'REZLIDHIA'
					WHEN REGIMEN LIKE '%IDHIFA%' THEN 'IDHIFA'
					WHEN REGIMEN LIKE '%MYLOTARG%' THEN 'MYLOTARG'
					WHEN REGIMEN LIKE '%DAURISMO%' THEN 'DAURISMO'
					WHEN REGIMEN LIKE '%ONUREG%' THEN 'HMA'    --ONUREG AND INQOVI ARE TREATED AS HMAs
					WHEN REGIMEN LIKE '%INQOVI%' THEN 'HMA'
					WHEN REGIMEN LIKE '%NEXAVAR%' THEN 'NEXAVAR'
					WHEN REGIMEN LIKE '%ETOPOSIDE%' THEN 'ETOPOSIDE'
					WHEN REGIMEN LIKE '%CLADRIBINE%' THEN 'CLADRIBINE'
					WHEN REGIMEN LIKE '%CLOFARABINE%' THEN 'CLOFARABINE'
					WHEN REGIMEN LIKE '%AZACITIDINE%' OR REGIMEN LIKE '%DECITABINE%' THEN 'HMA'
					WHEN REGIMEN LIKE '%L-DAC%' THEN  'L-DAC'
					WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 'ARSENIC_TRIOXIDE' 
					ELSE NULL
				END AS BACKBONE
				FROM {regimen_tbl_name}
									WHERE PATIENT_GID_RGMN IN
									(
										SELECT DISTINCT PATIENT_GID_RGMN
										FROM {patient_group}
										WHERE PATIENT_INTENSITY_GROUP = 'IC_ELIG'
									)
			)A
		)A
	)	
)
'''.format(ic_elig_final=ic_elig_final, regimen_tbl_name=regimen_tbl_name, patient_group=patient_group))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID_RGMN) 
FROM {ic_elig_final}
'''.format(ic_elig_final=ic_elig_final)).show()

# LOT TABLE FINAL - COMBINING IC ELIG AND INELIG PATIENTS

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {pat_comb_lot}'''.format(pat_comb_lot=pat_comb_lot))

In [ ]:
spark.sql('''
CREATE TABLE {pat_comb_lot} AS
(
	SELECT * 
    FROM
	(
		SELECT PATIENT_GID_RGMN AS PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_ELIG' AS TYPE
		FROM {ic_elig_final} -- SOME PATIENTS MAY HAVE ARSENIC REGIMENS

		UNION
		
		SELECT PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_INELIG' AS TYPE
		FROM {ic_inelig_final}
	)   
)
'''.format(ic_elig_final=ic_elig_final, ic_inelig_final=ic_inelig_final, pat_comb_lot=pat_comb_lot))

#### New Patient Intensity Rule (Based on Age + Product)

New Code Blocks added post this for the new column to be added

Comnbining all the regimens in the LoT 

In [ ]:
LOT_TBL = spark.sql(''' 
select * 
          from {pat_comb_lot}

'''.format(pat_comb_lot=pat_comb_lot))

LOT_TBL.createOrReplaceTempView('LOT_TBL')

In [ ]:
# Converting to Pandas dataframe

lot = LOT_TBL.toPandas()

df_lot = pd.DataFrame(lot)

# Function to merge regimens while keeping comma-separated format
def merge_regimens(regimens):
    # Split by comma, strip spaces, flatten, deduplicate
    merged = set()
    for regimen in regimens:
        for drug in regimen.split(","):
            merged.add(drug.strip())
    
    # Sort alphabetically and join
    return ", ".join(sorted(merged))

# Group by PATIENT_GID and LOT
LOT_REGIMEN_GROUP = (
    df_lot.groupby(["PATIENT_GID", "LOT"])
    .agg(
        LOT_START_DATE=("REGIMEN_START_DATE", "min"),
        LOT_END_DATE=("REGIMEN_END_DATE", "max"),
        LOT_REGIMEN_GROUP=("REGIMEN", merge_regimens),
    )
    .reset_index()
)

print(LOT_REGIMEN_GROUP)


In [ ]:
spark_df_LOT_REGIMEN_GROUP = spark.createDataFrame(LOT_REGIMEN_GROUP)
spark_df_LOT_REGIMEN_GROUP.createOrReplaceTempView("spark_df_LOT_REGIMEN_GROUP")

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {lot_regimen_group} ''')


spark.sql('''CREATE TABLE  {lot_regimen_group} AS
           (
          SELECT * FROM spark_df_LOT_REGIMEN_GROUP
          )
          ''')



First Dx and Tx Creation

In [ ]:
# Creating First Dx Tx table

fst_dx_tx_tbl = spark.sql(f'''
SELECT *
FROM
(
    SELECT A.PATIENT_GID,
           B.FIRST_DX,
           A.FIRST_TX,
           A.LAST_TX
    FROM
    (
        SELECT PATIENT_GID,
               MIN(TO_DATE(regimen_start_date)) AS FIRST_TX,
               MAX(TO_DATE(regimen_end_date)) AS LAST_TX
        FROM {pat_comb_lot}
        GROUP BY PATIENT_GID
    ) A
    LEFT JOIN
    (
        SELECT PATIENT_SK AS PATIENT_GID,
               MIN(TO_DATE(CLAIM_DATE)) AS FIRST_DX
        FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
        WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
        (
            '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
            'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
            'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
            'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
        )
        
        GROUP BY PATIENT_SK
    ) B
    ON A.PATIENT_GID = B.PATIENT_GID
                      
) A
''')

fst_dx_tx_tbl.createOrReplaceTempView('fst_dx_tx_tbl')


New Patient Intensity Assignment

In [ ]:
patient_intensity_updated_interim_tbl = spark.sql(''' 

SELECT 
    a.PATIENT_GID, 
    a.LOT_REGIMEN_GROUP, 
    a.LOT_START_DATE, 
    A.LOT_END_DATE,
    b.SOURCE_PATIENT_HIPAA_BIRTH_YEAR, 
    c.FIRST_DX
FROM (
    SELECT 
        PATIENT_GID, LOT_START_DATE, LOT_END_DATE, 
        LOT_REGIMEN_GROUP
    FROM {lot_regimen_group}
    WHERE LOT = 1
) a
LEFT JOIN (
    SELECT DISTINCT 
        PATIENT_SK, 
        SOURCE_PATIENT_HIPAA_BIRTH_YEAR
    FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
    WHERE source_flag = 'SHA_PTD'
      AND source_market = 'ONC'
      --AND market_code = 'VENC_AML'
      and patient_sk in (
    select patient_sk
    from patient_pool
)


) b 
    ON a.PATIENT_GID = b.PATIENT_SK
LEFT JOIN fst_dx_tx_tbl c 
    ON a.PATIENT_GID = c.PATIENT_GID;
                            

''')

patient_intensity_updated_interim_tbl.createOrReplaceTempView('patient_intensity_updated_interim_tbl')

In [ ]:
updated_patient_intensity = patient_intensity_updated_interim_tbl.toPandas()

In [ ]:


# Define product sets
IC_ELIG_PRODUCTS = {
    "MYLOTARG","RYDAPT","ARSENIC_TRIOXIDE","CLADRIBINE","CLOFARABINE","CYCLOPHOSPHAMIDE",
    "DAUNORUBICIN","ETOPOSIDE","FILGRASTIM","FLUDARABINE","HI-DAC","IDARUBICIN",
    "MITOXANTRONE","NEXAVAR","S-DAC","VINCRISTINE","VANFLYTA","VYXEOS"
}

FORCED_INELIG_PRODUCTS = {"AZACITIDINE","DECITABINE","ONUREG","INQOVI","TIBSOVO","IDHIFA"}

#import pandas as pd

def classify_patient(row):
    # Split regimen into product list
    products = {p.strip().upper() for p in row["LOT_REGIMEN_GROUP"].split(",")}
    
    # Age calculation if available
    birth_year = row["SOURCE_PATIENT_HIPAA_BIRTH_YEAR"]
    first_dx = row["FIRST_DX"]
    age = None
    if pd.notnull(birth_year) and pd.notnull(first_dx):
        # Convert birth_year to int if it's a string
        try:
            birth_year = int(birth_year)
        except ValueError:
            birth_year = None
        
        # Extract year from datetime if it's a date
        if hasattr(first_dx, "year"):
            dx_year = first_dx.year
        else:
            dx_year = int(first_dx)
        
        if birth_year is not None:
            age = dx_year - birth_year
    
    # Rule 1: If at least one IC_ELIG product → IC_ELIG
    if products & IC_ELIG_PRODUCTS:
        return "IC_ELIG"
    
    # Rule 2: If only HMA products → IC_INELIG
    if products.issubset(FORCED_INELIG_PRODUCTS):
        return "IC_INELIG"
    
    # Rule 3: If age present → age<=65 → IC_ELIG else IC_INELIG
    if age is not None:
        return "IC_ELIG" if age <= 65 else "IC_INELIG"
    
    # Rule 4: Default → IC_INELIG
    return "IC_INELIG"

# Apply classification
updated_patient_intensity["PATIENT_INTENSITY_UPDATED"] = updated_patient_intensity.apply(classify_patient, axis=1)

print(updated_patient_intensity)


In [ ]:
# Define backbone hierarchies
ic_elig_hierarchy = [
    "S-DAC","HI-DAC","VANFLYTA","VYXEOS","CYCLOPHOSPHAMIDE","VINCRISTINE","FLUDARABINE",
    "IDARUBICIN","DAUNORUBICIN","MITOXANTRONE","RYDAPT","VENCLEXTA","XOSPATA","TIBSOVO",
    "REZLIDHIA","IDHIFA","MYLOTARG","DAURISMO","ONUREG","INQOVI","NEXAVAR","ETOPOSIDE",
    "CLADRIBINE","CLOFARABINE","AZACITIDINE","DECITABINE","L-DAC","ARSENIC_TRIOXIDE"
]

ic_inelig_hierarchy = [
    "VENCLEXTA","TIBSOVO","REZLIDHIA","IDHIFA","XOSPATA","DAURISMO","L-DAC",
    "AZACITIDINE","DECITABINE","INQOVI","ONUREG"
]

# Convert hierarchies to dicts for fast lookup
ic_elig_rank = {drug: rank for rank, drug in enumerate(ic_elig_hierarchy, start=1)}
ic_inelig_rank = {drug: rank for rank, drug in enumerate(ic_inelig_hierarchy, start=1)}

def get_backbone(row):
    products = [p.strip() for p in row["LOT_REGIMEN_GROUP"].split(",")]
    if row["PATIENT_INTENSITY_UPDATED"] == "IC_ELIG":
        ranks = {p: ic_elig_rank[p] for p in products if p in ic_elig_rank}
    else:  # IC_INELIG
        ranks = {p: ic_inelig_rank[p] for p in products if p in ic_inelig_rank}
    if not ranks:  # no product found in hierarchy
        return "UNKNOWN"
    return min(ranks, key=ranks.get)

# Apply function
updated_patient_intensity["BACKBONE"] = updated_patient_intensity.apply(get_backbone, axis=1)

print(updated_patient_intensity)

In [ ]:
patient_intensity_based_on_age_n_product = spark.createDataFrame(updated_patient_intensity)
patient_intensity_based_on_age_n_product.createOrReplaceTempView("patient_intensity_based_on_age_n_product")

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {patient_intensity_based_on_age_n_product}''')

spark.sql('''
CREATE TABLE {patient_intensity_based_on_age_n_product} AS
(       
        SELECT * FROM patient_intensity_based_on_age_n_product
)         
          ''')